# Ausgrid Delivery 1: foundation pipeline orchestrator

This notebook is the primary way to run Delivery 1. It follows the same pattern as the `bms_sa_review` orchestrators: tested build modules underneath, a small slice first, visible checks after every stage, and an explicit full-run gate.

```text
inventory -> metadata + ID reconciliation -> duplicate audit
          -> canonical phase telemetry -> validation
```

**Delivery 1 does not calculate Volt-VAr or Volt-Watt conformance.** Those result builders and visual-analysis notebooks are planned in Deliveries 4 and 5.

## How to use this notebook

1. Install the project with the notebook extras.
2. Copy `config/analysis.example.toml` to `analysis.toml` and review it.
3. Run the sample cells from top to bottom.
4. Do not proceed past a failed assertion.
5. Review the conflicting-duplicate sample and canonical plots.
6. Use the full-run section only after the sample validation passes.

Raw parquet and Excel inputs are never modified. `OVERWRITE_SAMPLE` affects only outputs below the configured `derived_root`.

In [ ]:
from __future__ import annotations

import json
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

# Locate the downloaded project whether Jupyter starts in the project root
# or in the notebooks directory.
HERE = Path.cwd().resolve()
PROJECT_ROOT = next(
    (
        path
        for path in (HERE, *HERE.parents)
        if (path / "src" / "ausgrid_analysis").is_dir()
    ),
    None,
)
assert PROJECT_ROOT is not None, "Start Jupyter inside the ausgrid_analysis project."
sys.path.insert(0, str(PROJECT_ROOT / "src"))

from ausgrid_analysis.canonical import (
    build_canonical_phase,
    canonical_build_summary_path,
)
from ausgrid_analysis.config import load_config
from ausgrid_analysis.db import (
    canonical_output_path,
    connect,
    duplicate_audit_path,
    duplicate_summary_path,
)
from ausgrid_analysis.duplicates import conflicting_rows_path, run_duplicate_audit
from ausgrid_analysis.inventory import inventory_output_path, run_inventory
from ausgrid_analysis.metadata import (
    metadata_output_path,
    metadata_summary_path,
    prepare_metadata,
    reconciliation_output_path,
)
from ausgrid_analysis.schemas import sql_string
from ausgrid_analysis.validation import validate_canonical_phase

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 100)
plt.style.use("seaborn-v0_8-whitegrid")

In [ ]:
# User-controlled settings.
CONFIG_PATH = PROJECT_ROOT / "analysis.toml"
SAMPLE_MONTH = "2025-04"
SAMPLE_BUCKET = 0
OVERWRITE_SAMPLE = False  # Set True only when intentionally rebuilding the sample.

assert CONFIG_PATH.is_file(), (
    "Copy config/analysis.example.toml to analysis.toml and review it first."
)

config = load_config(CONFIG_PATH, check_inputs=True)
scope = config.scope(SAMPLE_MONTH, SAMPLE_BUCKET)

def read_json(path: Path) -> dict:
    return json.loads(path.read_text(encoding="utf-8"))

def query(sql: str) -> pd.DataFrame:
    connection = connect(config)
    try:
        return connection.execute(sql).fetchdf()
    finally:
        connection.close()

display(pd.DataFrame({
    "setting": [
        "sample scope", "telemetry", "metadata", "derived outputs",
        "active export sign", "reactive absorption sign", "source timezone",
    ],
    "value": [
        scope.label,
        str(config.paths.telemetry_parquet),
        str(config.paths.metadata_workbook),
        str(config.paths.derived_root),
        config.assumptions.active_export_sign,
        config.assumptions.reactive_absorbing_sign,
        config.assumptions.source_timezone,
    ],
}))

## Stage 0 — source inventory

This is a read-only scan of the selected month and site bucket. Check schema, row count, time coverage, phases and source files before creating derived tables.

In [ ]:
inventory = run_inventory(config, scope)

display(pd.DataFrame([inventory["overall"]]))
display(pd.DataFrame(inventory["by_source_month"]))
display(pd.DataFrame(inventory["by_phase"]))
display(pd.DataFrame(inventory["telemetry_schema"])[["column_name", "column_type", "null"]])

In [ ]:
# Gate 0: do not continue if source identifiers or timestamps are unusable.
inventory_gate = pd.DataFrame({
    "check": ["rows found", "serials found", "no null timestamps", "no blank serials"],
    "passed": [
        inventory["overall"]["n_rows"] > 0,
        inventory["overall"]["n_serials"] > 0,
        inventory["overall"]["null_timestamps"] == 0,
        inventory["overall"]["null_or_blank_serials"] == 0,
    ],
})
display(inventory_gate)
assert inventory_gate["passed"].all(), "Stage 0 failed; inspect the inventory above."

## Stage 1 — canonical metadata and scoped ID reconciliation

The metadata workbook is normalised once. ID reconciliation uses the same sample scope, so this first run does not scan the full 303-million-row parquet. Missing metadata is reported but does not remove telemetry.

In [ ]:
metadata_report_path = metadata_summary_path(config, scope)
reconciliation_path = reconciliation_output_path(config, scope)

if metadata_report_path.exists() and reconciliation_path.exists() and not OVERWRITE_SAMPLE:
    metadata_report = read_json(metadata_report_path)
    print("Reusing existing scoped metadata report. Set OVERWRITE_SAMPLE=True to rebuild.")
else:
    metadata_report = prepare_metadata(config, scope, overwrite=OVERWRITE_SAMPLE)

display(pd.DataFrame([metadata_report["reconciliation_counts"]]).fillna(0))
display(pd.DataFrame([metadata_report["cohort_counts"]]).fillna(0))

reconciliation = pd.read_csv(reconciliation_path, dtype={"serial": "string"})
display(reconciliation.groupby("reconciliation_status", dropna=False).size().rename("n_ids").to_frame())
display(reconciliation[reconciliation["reconciliation_status"] != "matched"].head(20))

In [ ]:
metadata_path = metadata_output_path(config)
metadata_quality = query(f"""
    SELECT
        analysis_cohort,
        install_phase_count,
        count(*) AS n_sites,
        count_if(solar_capacity_kw IS NULL) AS missing_solar_capacity,
        count_if(s_rated_kva IS NULL) AS missing_s_rated
    FROM read_parquet({sql_string(metadata_path)})
    GROUP BY analysis_cohort, install_phase_count
    ORDER BY analysis_cohort, install_phase_count
""")
display(metadata_quality)

cohort_counts = metadata_quality.groupby("analysis_cohort")["n_sites"].sum()
cohort_counts.plot(kind="bar", title="Metadata cohorts", ylabel="sites", rot=0, figsize=(7, 3));
plt.show()

In [ ]:
# Gate 1: metadata IDs must be unique; unavailable ratings must remain explicit.
metadata_gate = query(f"""
    SELECT
        count(*) AS n_rows,
        count(DISTINCT serial) AS n_serials,
        count_if(serial IS NULL OR trim(serial) = '') AS blank_serials,
        count_if(s_rated_kva IS NOT NULL) AS inferred_ratings
    FROM read_parquet({sql_string(metadata_path)})
""")
display(metadata_gate)
assert int(metadata_gate.loc[0, "n_rows"]) == int(metadata_gate.loc[0, "n_serials"])
assert int(metadata_gate.loc[0, "blank_serials"]) == 0
assert int(metadata_gate.loc[0, "inferred_ratings"]) == 0

## Stage 2 — duplicate-key audit

The candidate key is `(serial, MeasureTime, Vphase)`. Identical duplicates can be collapsed. Conflicting duplicates are quarantined and must be inspected rather than averaged.

In [ ]:
duplicate_report_path = duplicate_summary_path(config, scope)
audit_path = duplicate_audit_path(config, scope)

if duplicate_report_path.exists() and audit_path.exists() and not OVERWRITE_SAMPLE:
    duplicate_report = read_json(duplicate_report_path)
    print("Reusing existing duplicate audit. Set OVERWRITE_SAMPLE=True to rebuild.")
else:
    duplicate_report = run_duplicate_audit(config, scope, overwrite=OVERWRITE_SAMPLE)

duplicate_summary = pd.DataFrame({
    "metric": [
        "duplicate keys", "identical keys", "conflicting keys",
        "identical rows collapsed", "conflicting rows quarantined",
        "cross-file duplicate keys",
    ],
    "value": [
        duplicate_report["duplicate_keys"],
        duplicate_report["identical_duplicate_keys"],
        duplicate_report["conflicting_duplicate_keys"],
        duplicate_report["identical_rows_to_collapse"],
        duplicate_report["conflicting_rows_to_quarantine"],
        duplicate_report["cross_file_duplicate_keys"],
    ],
})
display(duplicate_summary)

In [ ]:
duplicate_classes = query(f"""
    SELECT duplicate_class, count(*) AS n_keys, sum(row_count) AS n_source_rows
    FROM read_parquet({sql_string(audit_path)})
    GROUP BY duplicate_class
    ORDER BY duplicate_class
""")
display(duplicate_classes)
if not duplicate_classes.empty:
    duplicate_classes.set_index("duplicate_class")["n_keys"].plot(
        kind="bar", title="Repeated keys by classification", ylabel="keys", rot=0, figsize=(7, 3)
    )
    plt.show()

conflict_path = conflicting_rows_path(config, scope)
conflict_preview = query(f"""
    SELECT *
    FROM read_parquet({sql_string(conflict_path)})
    ORDER BY serial, measure_time, phase, source_file
    LIMIT 50
""")
display(conflict_preview)

In [ ]:
# Gate 2: the duplicate audit itself must account for its classifications.
assert duplicate_report["duplicate_keys"] == (
    duplicate_report["identical_duplicate_keys"]
    + duplicate_report["conflicting_duplicate_keys"]
)
assert duplicate_report["conflicting_rows_to_quarantine"] >= (
    2 * duplicate_report["conflicting_duplicate_keys"]
)
print("Stage 2 gate passed. Review conflict_preview before continuing.")

## Stage 3 — canonical phase telemetry

This writes one row per accepted `(serial, timestamp, phase)` key. Raw P/Q are retained alongside provisional normalised signs, UTC/local timestamps, metadata flags and duplicate provenance.

In [ ]:
canonical_report_path = canonical_build_summary_path(config, scope)
canonical_dir = canonical_output_path(config, scope)

if canonical_report_path.exists() and canonical_dir.exists() and not OVERWRITE_SAMPLE:
    canonical_report = read_json(canonical_report_path)
    print("Reusing existing canonical sample. Set OVERWRITE_SAMPLE=True to rebuild.")
else:
    canonical_report = build_canonical_phase(config, scope, overwrite=OVERWRITE_SAMPLE)

display(pd.DataFrame([canonical_report]))
canonical_glob = str(canonical_dir / "**" / "*.parquet")

In [ ]:
canonical_quality = query(f"""
    SELECT
        phase,
        duplicate_status,
        metadata_available,
        count(*) AS n_rows,
        count(DISTINCT serial) AS n_serials,
        count_if(row_has_null_measurement) AS null_measurement_rows,
        count_if(NOT voltage_physical_ok) AS voltage_flagged_rows
    FROM read_parquet({sql_string(canonical_glob)}, hive_partitioning = true)
    GROUP BY phase, duplicate_status, metadata_available
    ORDER BY phase, duplicate_status, metadata_available
""")
display(canonical_quality)

canonical_preview = query(f"""
    SELECT
        serial, timestamp_utc, timestamp_local, phase, voltage_v,
        active_power_raw_w, p_export_w,
        reactive_power_raw_var, q_absorbing_var, q_generator_var,
        duplicate_status, metadata_available, analysis_cohort
    FROM read_parquet({sql_string(canonical_glob)}, hive_partitioning = true)
    ORDER BY serial, timestamp_utc, phase
    LIMIT 30
""")
display(canonical_preview)

In [ ]:
# Deterministic, bounded visual sample. This does not load the whole scope into pandas.
plot_sample = query(f"""
    SELECT phase, voltage_v, p_export_w, q_generator_var
    FROM read_parquet({sql_string(canonical_glob)}, hive_partitioning = true)
    WHERE mod(hash(serial, timestamp_utc, phase), 100) = 0
      AND voltage_v IS NOT NULL
      AND p_export_w IS NOT NULL
      AND q_generator_var IS NOT NULL
    LIMIT 30000
""")

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for phase, frame in plot_sample.groupby("phase"):
    axes[0].scatter(frame["voltage_v"], frame["p_export_w"], s=6, alpha=0.25, label=phase)
    axes[1].scatter(frame["voltage_v"], frame["q_generator_var"], s=6, alpha=0.25, label=phase)
axes[0].set(title="Active power versus voltage", xlabel="Voltage (V)", ylabel="P export (W)")
axes[1].set(title="Reactive power versus voltage", xlabel="Voltage (V)", ylabel="Q generator convention (VAr)")
for axis in axes:
    axis.legend(title="phase")
plt.tight_layout()
plt.show()

In [ ]:
# Gate 3: uniqueness and timestamp conversion must hold before final validation.
canonical_gate = query(f"""
    WITH canonical AS (
        SELECT * FROM read_parquet({sql_string(canonical_glob)}, hive_partitioning = true)
    )
    SELECT
        count(*) AS n_rows,
        count_if(timestamp_utc IS NULL) AS null_utc,
        count_if(timestamp_local IS NULL) AS null_local,
        count(*) - count(DISTINCT (serial, timestamp_utc, phase)) AS duplicate_rows,
        count_if(p_export_w != active_power_raw_w * {config.assumptions.active_export_sign}) AS active_sign_errors,
        count_if(q_absorbing_var != reactive_power_raw_var * {config.assumptions.reactive_absorbing_sign}) AS reactive_sign_errors
    FROM canonical
""")
display(canonical_gate)
assert int(canonical_gate.loc[0, "null_utc"]) == 0
assert int(canonical_gate.loc[0, "null_local"]) == 0
assert int(canonical_gate.loc[0, "duplicate_rows"]) == 0
assert int(canonical_gate.loc[0, "active_sign_errors"]) == 0
assert int(canonical_gate.loc[0, "reactive_sign_errors"]) == 0

## Stage 4 — exact row accounting and final sample validation

The expected canonical count is:

`source rows - collapsed identical rows - all quarantined conflicting rows`

In [ ]:
validation = validate_canonical_phase(config, scope)

validation_headline = {
    key: value
    for key, value in validation.items()
    if key not in {"monthly_coverage", "scope", "failures"}
}
display(pd.DataFrame([validation_headline]).T.rename(columns={0: "value"}))
display(pd.DataFrame(validation["monthly_coverage"]))

assert validation["status"] == "pass"
assert validation["accounting_difference"] == 0
assert validation["duplicate_keys_remaining"] == 0

In [ ]:
# Final sample decision table.
sample_decision = pd.DataFrame({
    "gate": [
        "source inventory", "metadata uniqueness", "duplicate accounting",
        "canonical uniqueness", "final row accounting",
    ],
    "passed": [
        bool(inventory_gate["passed"].all()),
        int(metadata_gate.loc[0, "n_rows"]) == int(metadata_gate.loc[0, "n_serials"]),
        duplicate_report["duplicate_keys"] == duplicate_report["identical_duplicate_keys"] + duplicate_report["conflicting_duplicate_keys"],
        int(canonical_gate.loc[0, "duplicate_rows"]) == 0,
        validation["status"] == "pass" and validation["accounting_difference"] == 0,
    ],
})
display(sample_decision)
assert sample_decision["passed"].all()
print("Sample foundation pipeline passed. Review the tables and plots before widening scope.")

# Optional full-dataset build

Do not run this section merely because the sample assertions passed. First inspect:

- telemetry-only and metadata-only IDs;
- conflicting duplicate examples;
- phase and voltage distributions;
- provisional P/Q sign behavior;
- available disk space under `derived_root`.

The full build may take substantial time and temporary disk. The following exact confirmation prevents accidental execution.

In [ ]:
FULL_RUN_CONFIRMATION = ""  # Change to: RUN FULL DATASET
OVERWRITE_FULL = False

assert FULL_RUN_CONFIRMATION == "RUN FULL DATASET", (
    "Full run remains locked. Review the complete sample first."
)
full_scope = config.scope(None, None)
print("Full run unlocked:", full_scope.label)

## Full Stage 0 — inventory

In [ ]:
full_inventory = run_inventory(config, full_scope)
display(pd.DataFrame([full_inventory["overall"]]))
display(pd.DataFrame(full_inventory["by_source_month"]))
assert full_inventory["overall"]["null_timestamps"] == 0
assert full_inventory["overall"]["null_or_blank_serials"] == 0

## Full Stage 1 — metadata and complete ID reconciliation

In [ ]:
full_metadata_summary_path = metadata_summary_path(config, full_scope)
full_reconciliation_path = reconciliation_output_path(config, full_scope)
if full_metadata_summary_path.exists() and full_reconciliation_path.exists() and not OVERWRITE_FULL:
    full_metadata = read_json(full_metadata_summary_path)
else:
    full_metadata = prepare_metadata(config, full_scope, overwrite=OVERWRITE_FULL)
display(pd.DataFrame([full_metadata["reconciliation_counts"]]).fillna(0))
display(pd.read_csv(full_reconciliation_path).query("reconciliation_status != 'matched'").head(100))

## Full Stage 2 — duplicate audit

In [ ]:
full_duplicate_summary_path = duplicate_summary_path(config, full_scope)
if full_duplicate_summary_path.exists() and duplicate_audit_path(config, full_scope).exists() and not OVERWRITE_FULL:
    full_duplicates = read_json(full_duplicate_summary_path)
else:
    full_duplicates = run_duplicate_audit(config, full_scope, overwrite=OVERWRITE_FULL)
display(pd.DataFrame([full_duplicates]).T.rename(columns={0: "value"}))
assert full_duplicates["duplicate_keys"] == full_duplicates["identical_duplicate_keys"] + full_duplicates["conflicting_duplicate_keys"]

## Full Stage 3 — canonical phase telemetry

In [ ]:
full_canonical_summary_path = canonical_build_summary_path(config, full_scope)
if full_canonical_summary_path.exists() and canonical_output_path(config, full_scope).exists() and not OVERWRITE_FULL:
    full_canonical = read_json(full_canonical_summary_path)
else:
    full_canonical = build_canonical_phase(config, full_scope, overwrite=OVERWRITE_FULL)
display(pd.DataFrame([full_canonical]).T.rename(columns={0: "value"}))

## Full Stage 4 — validation

In [ ]:
full_validation = validate_canonical_phase(config, full_scope)
display(pd.DataFrame([{k: v for k, v in full_validation.items() if k != "monthly_coverage"}]).T.rename(columns={0: "value"}))
display(pd.DataFrame(full_validation["monthly_coverage"]))
assert full_validation["status"] == "pass"
assert full_validation["accounting_difference"] == 0
assert full_validation["duplicate_keys_remaining"] == 0
print("Delivery 1 full foundation build passed.")

## Delivery 1 complete

The next notebook will consume `canonical_phase` and build structured site-interval features. Volt-VAr and Volt-Watt result tables are not produced until their observability, phase aggregation, rating and irradiance dependencies have been made explicit.